# Proteomics QC Pipeline — Hands-On Tutorial

**Course: Proteomics Data Analysis**  
**Environment: Google Colab (or local Jupyter)**

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and inspect a DIA proteomics intensity matrix
2. Apply protein completeness filtering and explain the trade-offs
3. Normalise intensities using median scaling and understand why it works
4. Detect and confirm technical outlier samples using multiple methods
5. Diagnose and correct plate batch effects using ComBat or plate-median correction
6. Interpret coefficient of variation as a measure of assay reproducibility
7. Generate a standardised QC report ready for publication

---

## Dataset

We use a **synthetic dataset** with the following known properties:
- 2 000 proteins, 160 samples across 4 plates (40 samples/plate)
- 80 Cases / 80 Controls (balanced)
- **Batch effects** baked in: plates 2–4 have intensity shifts of +0.35, -0.25, +0.50 log2 units
- **4 outlier samples** hidden in the data (you will find them!)
- ~37 % missing values (realistic for DIA plasma proteomics)

---
## 0. Environment Setup

Run the cell below once to install dependencies and clone the pipeline repository.
Skip if running locally with the environment already set up.

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────────
# This takes ~60–90 seconds on Colab's first run
import subprocess, sys

packages = [
    "pandas>=2.0",
    "numpy>=1.24",
    "matplotlib>=3.7",
    "scipy>=1.10",
    "scikit-learn>=1.3",
    "pyarrow>=12.0",
    "inmoose>=0.3",   # ComBat batch correction
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
print("✓ Dependencies installed")

# ── Clone pipeline repo ─────────────────────────────────────────────────────
import os
if not os.path.exists("dea-proteomics-course"):
    subprocess.check_call(["git", "clone", "-q",
        "https://github.com/nigelkurgan/ddea-proteomics-course.git"])
os.chdir("dea-proteomics-course")
sys.path.insert(0, ".")
print("✓ Repository ready")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 110

from pathlib import Path

# Pipeline modules
from proteomics_qc.proteomics.filters import filter_by_completeness, completeness_at_thresholds
from proteomics_qc.proteomics.normalise import median_scaling, impute_knn, plate_median_correction, combat_correction
from proteomics_qc.proteomics.outliers import (
    sample_mean_outliers, missing_rate_outliers, PCAOutliers,
    density_outliers, ks_confirm_outliers, build_outlier_table
)
from proteomics_qc.proteomics.batch import plate_distance_stats, pc_factor_associations
from proteomics_qc.plots.style import apply_style, PALETTE
from proteomics_qc.plots.distribution import plot_sample_boxplot, plot_density_per_sample, plot_protein_rank_abundance
from proteomics_qc.plots.missing import plot_missing_heatmap, plot_missing_by_threshold, plot_missing_per_group
from proteomics_qc.plots.batch_effects import plot_pca, plot_plate_distances, plot_pc_factor_heatmap, plot_correlation_heatmap
from proteomics_qc.plots.cv import compute_sample_cv, compute_intra_cv, compute_inter_cv, plot_cv_violin, plot_cv_comparison

apply_style()
print("✓ All imports successful")

---
## Stage 1: Load the Data

### What format does proteomics data come in?

Spectronaut (and DIA-NN) export results as a **long-format report** or a **matrix**.  
For QC, we use the **protein-level matrix**:

```
          Sample_001  Sample_002  ...  Sample_160
P00001        23.4        22.8   ...      24.1
P00002        19.1         NaN   ...      20.3
  ...          ...         ...   ...       ...
```

- Rows = proteins (identified by UniProt accession)
- Columns = samples
- Values = **log2 MS1 intensity** (LFQ or MaxLFQ)
- **NaN** = protein not detected in that sample

We store this in [Apache Parquet](https://parquet.apache.org/) format — columnar storage that is fast to read and compact.

In [ ]:
# ── Generate demo data (skip if already generated) ──────────────────────────
demo_parquet = Path("data/demo/demo_proteomics.parquet")
demo_meta_csv = Path("data/demo/demo_metadata.csv")

if not demo_parquet.exists():
    import subprocess
    subprocess.check_call(["python", "scripts/generate_demo_data.py"])

# ── Load protein matrix (proteins × samples parquet) ────────────────────────
raw = pd.read_parquet(demo_parquet)

# Separate protein metadata columns (PG_*) from intensity columns
meta_cols   = [c for c in raw.columns if c.startswith("PG_")]
sample_cols = [c for c in raw.columns if not c.startswith("PG_")]

prot_meta = raw.set_index("PG_ProteinAccessions")[meta_cols[1:]]

# TRANSPOSE to (samples × proteins) — most analysis functions expect this orientation
quant_df = raw.set_index("PG_ProteinAccessions")[sample_cols].T

print(f"Matrix shape: {quant_df.shape[0]} samples × {quant_df.shape[1]} proteins")
print(f"Intensity range (log2): {quant_df.min().min():.1f} – {quant_df.max().max():.1f}")
print(f"Overall missing rate: {quant_df.isna().mean().mean():.1%}")
quant_df.head(3)

In [ ]:
# ── Load sample metadata ─────────────────────────────────────────────────────
sample_meta = pd.read_csv(demo_meta_csv, index_col=0)
sample_meta = sample_meta.reindex(quant_df.index)  # align to matrix row order

print("Sample metadata:")
print(sample_meta.value_counts(["plate", "group"]).to_string())
sample_meta.head()

---
## Stage 2: Missing Value Exploration

### Why do proteomics data have missing values?

Missing values in DIA proteomics are **not random** — they are **Missing Not At Random (MNAR)**:  
low-abundance proteins fall below the instrument's detection limit and are not confidently identified.

**Key questions before filtering:**
1. Which proteins are most frequently missing?
2. Is missingness structured by plate (= technical) or by sample group (= biological)?
3. What completeness threshold balances protein count vs. data quality?

The heatmap below shows proteins (x-axis) × samples (y-axis). Black = missing.

In [ ]:
# Missing value heatmap — samples sorted by plate to reveal structure
plot_missing_heatmap(quant_df, sample_meta, sort_by="plate")

In [ ]:
# How many proteins survive different completeness thresholds?
# Use this to choose your filtering threshold
plot_missing_by_threshold(quant_df)

# Print the numbers
counts = completeness_at_thresholds(quant_df)
print("\nProteins retained at each threshold:")
print(counts.to_string())

In [ ]:
# Is missingness uniform across plates, or does one plate have more missing?
plot_missing_per_group(quant_df, sample_meta["plate"])

---
## Stage 3: Protein Completeness Filtering

We remove proteins detected in fewer than `threshold` fraction of samples.  
The default is **20 %** — a commonly used cutoff for longitudinal DIA studies.

> **Exercise 3.1**: Change the threshold below to 0.10 and 0.50 and note how many proteins are retained.  
> Which threshold would you choose for a study with 2 groups where one group has systematically lower expression?

In [ ]:
COMPLETENESS_THRESHOLD = 0.20  # ← try changing this

quant_filt = filter_by_completeness(quant_df, threshold=COMPLETENESS_THRESHOLD, axis="features")
print(f"\nRemaining: {quant_filt.shape[1]} proteins in {quant_filt.shape[0]} samples")

---
## Stage 4: Normalisation

### Why normalise?

Even after careful sample preparation, total protein input, injection volume, and instrument sensitivity vary between samples.  This introduces **systematic offsets** unrelated to biology.

**Median scaling** corrects for this in two steps:
1. Shift each sample's median to the global median (**loading correction**)
2. Equalise the spread across samples (**MAV normalisation**)

After normalisation, the per-sample boxplot should show **aligned medians**.

> **Exercise 4.1**: Look at the boxplot *before* normalisation (raw data, first plot) and *after* (second plot).  
> Can you see which samples were shifted? Do you notice any persistent outliers?

In [ ]:
# Before normalisation
print("Before normalisation:")
plot_sample_boxplot(quant_filt, sample_meta, color_by="plate")

In [ ]:
# Apply median scaling
quant_norm = median_scaling(quant_filt)

print("After median scaling:")
plot_sample_boxplot(quant_norm, sample_meta, color_by="plate")

In [ ]:
# KNN imputation for PCA / clustering (complete matrix required)
# IMPORTANT: Use this ONLY for visualisation — not for downstream statistics
print("Applying KNN imputation (for PCA/clustering only)...")
quant_imputed = impute_knn(quant_norm, n_neighbors=5)
print(f"  Imputed matrix: {quant_imputed.shape} (fully observed: {quant_imputed.isna().sum().sum()} NaN)")

---
## Stage 5: Outlier Detection

### Strategy: multiple independent metrics + KS-test confirmation

We use **5 detection methods** in parallel:

| Method | What it catches |
|--------|-----------------|
| Mean intensity Z-score | Globally shifted samples (degraded / contaminated) |
| Missing rate Z-score | Failed injections (unusually sparse) |
| KDE density Z-score | Samples with unusual intensity distributions |
| PCA Euclidean distance | Samples far from the cohort centroid in PC space |
| PCA Mahalanobis distance | Samples deviating in subtle, low-variance directions |

A **KS test** then confirms each candidate: the sample's intensity distribution must be
significantly different from the rest of the cohort (p < 0.05).

> **Exercise 5.1**: The demo data has **4 hidden outliers**. After running the detection cell,  
> look at the `outlier_summary` table. Can you identify all 4?  
> Which method caught each one?

In [ ]:
ZSCORE_THRESHOLD = 2.576  # 99th percentile, α ≈ 0.01

# Run all detection methods
print("Running outlier detection...")
outlier_mean    = sample_mean_outliers(quant_norm, ZSCORE_THRESHOLD)
outlier_missing = missing_rate_outliers(quant_filt, ZSCORE_THRESHOLD)
outlier_density = density_outliers(quant_norm, ZSCORE_THRESHOLD)

pca_out = PCAOutliers(quant_imputed, n_components=5, threshold=ZSCORE_THRESHOLD)

detection_results = {
    "Mean intensity":  outlier_mean,
    "Missing rate":    outlier_missing,
    "Density":         outlier_density,
    "PCA-Euclidean":   pca_out.euclidean_outliers,
    "PCA-Mahalanobis": pca_out.mahalanobis_outliers,
}

print("\nFlagged by each method:")
for method, flagged in detection_results.items():
    print(f"  {method:20s}: {len(flagged)} samples")

In [ ]:
# Pool all candidates and confirm with KS test
all_candidates = sorted({s for lst in detection_results.values() for s in lst})
print(f"Total candidate outliers: {len(all_candidates)}")

ks_confirmed = ks_confirm_outliers(quant_norm, all_candidates, alpha=0.05)
outlier_table = build_outlier_table(quant_norm.index.tolist(), detection_results, ks_confirmed)

print(f"\nKS-confirmed outliers: {outlier_table['KS_confirmed'].eq('✓').sum()}")
print()
outlier_table

In [ ]:
# Visualise: density curves highlight outliers in red
confirmed_outliers = [s for s, v in ks_confirmed.items() if v]
plot_density_per_sample(quant_norm, outlier_samples=confirmed_outliers)

In [ ]:
# PCA coloured by plate — outliers marked with ×
plot_pca(quant_imputed, sample_meta, color_by="plate",
         title="PCA — plate (pre-correction, outliers marked ×)",
         outlier_samples=confirmed_outliers)

---
## Stage 6: Batch-Effect QC

### What are batch effects?

In multi-plate DIA experiments, each MS plate (= set of samples run on the same day/instrument)  
introduces a **systematic offset** unrelated to biology.  If uncorrected:
- PC1 will separate plates, not biology
- Differential expression tests will have inflated false positives
- Clustering will group by plate, not phenotype

### How do we diagnose batch effects?

1. **PCA coloured by plate** — do samples cluster by plate?
2. **Within vs. between plate distances** — within << between = strong batch effect
3. **PC × factor heatmap** — does plate explain PC1/PC2 more than biology?

> **Exercise 6.1**: Look at the PC × factor heatmap.  
> Which factor drives PC1: plate or group? What does this tell you about the data?

In [ ]:
# Build clean (outlier-removed) imputed matrix for batch QC
confirmed_set = set(confirmed_outliers)
quant_imputed_clean = quant_imputed.loc[
    [s for s in quant_imputed.index if s not in confirmed_set]
]
meta_clean = sample_meta.reindex(quant_imputed_clean.index)
print(f"Clean matrix: {quant_imputed_clean.shape[0]} samples ({len(confirmed_outliers)} outliers removed)")

# PCA coloured by plate
plot_pca(quant_imputed_clean, meta_clean, color_by="plate",
         title="PCA — by plate (outliers removed, pre-correction)")

In [ ]:
# PCA coloured by group — is biology visible?
plot_pca(quant_imputed_clean, meta_clean, color_by="group",
         title="PCA — by group (outliers removed, pre-correction)")

In [ ]:
# Within vs. between plate distances
plate_labels = sample_meta["plate"]
dist_stats = plate_distance_stats(quant_imputed_clean, plate_labels.reindex(quant_imputed_clean.index),
                                   n_pca_components=10)
print(f"KS p-value (within vs. between plate): {dist_stats['ks_pvalue']:.4g}")
if dist_stats["ks_pvalue"] < 0.05:
    print("→ Significant plate separation detected — batch correction recommended")

plot_plate_distances(dist_stats["within"], dist_stats["between"], dist_stats["ks_pvalue"])

In [ ]:
# PC × factor associations
pc_assoc = pc_factor_associations(
    quant_imputed_clean, meta_clean,
    factors=["plate", "group"],
    n_pca_components=10
)
print("PC × factor associations (-log10 p-value):")
print(pc_assoc.round(2).to_string())
plot_pc_factor_heatmap(pc_assoc)

### Batch Correction

We compare two approaches:

| Method | How | When to use |
|--------|-----|-------------|
| **Plate-median** | Subtract plate protein-median, add global median | Simple, works with NaN |
| **ComBat** | Bayesian parametric model (additive + multiplicative) | More powerful; needs complete matrix |

> **Exercise 6.2**: Run both correction methods below and compare the PCA plots.  
> Does the plate structure disappear? Does the group separation improve?

In [ ]:
plate_lbl = sample_meta["plate"].reindex(quant_imputed_clean.index)

# ── Method 1: Plate-median correction (works with missing values) ────────────
quant_pm = plate_median_correction(quant_imputed_clean, plate_lbl)
plot_pca(quant_pm, meta_clean, color_by="plate",
         title="PCA — plate-median corrected")
plot_pca(quant_pm, meta_clean, color_by="group",
         title="PCA — group (plate-median corrected)")

In [ ]:
# ── Method 2: ComBat (Bayesian, requires complete matrix) ────────────────────
try:
    quant_combat = combat_correction(quant_imputed_clean, plate_lbl)
    plot_pca(quant_combat, meta_clean, color_by="plate",
             title="PCA — ComBat corrected")
    plot_pca(quant_combat, meta_clean, color_by="group",
             title="PCA — group (ComBat corrected)")
    print("\nComBat correction applied successfully")
except Exception as e:
    print(f"ComBat failed: {e}")
    print("Using plate-median correction instead")
    quant_combat = quant_pm

---
## Stage 7: Coefficient of Variation (CV) Analysis

### What is CV and why does it matter?

**CV = (standard deviation / mean) × 100 %**

In proteomics, a low CV means a protein is quantified reproducibly across samples.

| CV type | What it measures | Typical benchmark |
|---------|------------------|-----------------|
| Intra-individual | Technical reproducibility (within the same biological sample) | < 20 % |
| Inter-individual | Biological variability (between different samples/subjects) | > Intra |

If **inter-CV ≤ intra-CV**, the assay cannot distinguish individuals — the technical noise  
swamps the biological signal.

> **Exercise 7.1**: Look at the CV comparison plot. Is the inter-individual CV larger than  
> the intra-individual CV? What does this tell you about the assay's power to detect  
> differences between Case and Control samples?

In [ ]:
# Convert log2 → linear for CV calculation
# (CV is meaningless in log-space; must use raw intensities)
quant_linear = 2 ** quant_norm.fillna(float("nan"))

overall_cv = compute_sample_cv(quant_linear)
# Use plate as the grouping variable for intra/inter CV
# In a real study, use subject_id for intra-individual CV
intra_cv = compute_intra_cv(quant_linear, sample_meta["plate"].reindex(quant_linear.index))
inter_cv  = compute_inter_cv(quant_linear, sample_meta["plate"].reindex(quant_linear.index))

print(f"Median overall CV : {overall_cv.median():.1f}%")
print(f"Median intra CV   : {intra_cv.median():.1f}%"  if not intra_cv.empty else "Intra CV: N/A")
print(f"Median inter CV   : {inter_cv.median():.1f}%"  if not inter_cv.empty else "Inter CV: N/A")

plot_cv_violin(overall_cv)

In [ ]:
if not intra_cv.empty and not inter_cv.empty:
    plot_cv_comparison(intra_cv, inter_cv)

---
## Stage 8: Summary & Analysis-Ready Output

We now produce a clean, analysis-ready protein matrix:
1. Remove confirmed outlier samples
2. Apply batch correction
3. Save as parquet (same format as input, for compatibility with downstream tools)

In [ ]:
# Decision: exclude samples flagged by ≥3 methods (adjust as needed)
N_METHODS_THRESHOLD = 3

if not outlier_table.empty:
    remove_samples = outlier_table[
        outlier_table["n_methods_flagged"] >= N_METHODS_THRESHOLD
    ].index.tolist()
else:
    remove_samples = []

print(f"Samples to remove: {len(remove_samples)}")
print(f"  {remove_samples}")

quant_clean = quant_norm.drop(index=remove_samples, errors="ignore")
print(f"\nFinal matrix (normalised, outliers removed): {quant_clean.shape}")

In [ ]:
import json

# Save analysis-ready output
Path("results/qc/tables").mkdir(parents=True, exist_ok=True)

# Reconstruct proteins × samples format (matches input)
out_df = prot_meta.join(quant_clean.T, how="left").reset_index()
out_df.to_parquet("results/qc/proteomics_analysis_ready.parquet", index=False)
print("Saved: results/qc/proteomics_analysis_ready.parquet")

# Save QC decisions for reproducibility
qc_decisions = {
    "normalization": "median_scaling",
    "completeness_threshold": COMPLETENESS_THRESHOLD,
    "outlier_n_methods_threshold": N_METHODS_THRESHOLD,
    "n_outliers_removed": len(remove_samples),
    "outliers_removed": remove_samples,
    "n_samples_final": int(quant_clean.shape[0]),
    "n_proteins_final": int(quant_clean.shape[1]),
}
with open("results/qc/tables/qc_decisions.json", "w") as f:
    json.dump(qc_decisions, f, indent=2)
print("Saved: results/qc/tables/qc_decisions.json")
print()
print("=== QC Summary ===")
print(f"  Input:  {quant_df.shape[0]} samples × {quant_df.shape[1]} proteins")
print(f"  After filtering:  {quant_filt.shape[1]} proteins (threshold={COMPLETENESS_THRESHOLD:.0%})")
print(f"  After outlier removal: {quant_clean.shape[0]} samples")
print(f"  Final: {quant_clean.shape[0]} samples × {quant_clean.shape[1]} proteins")

---
## Extended Exercises

### Exercise A — Threshold sensitivity
Change `COMPLETENESS_THRESHOLD` to 0.05, 0.50, and 1.0.  
Run the full pipeline for each.  How many proteins remain at each threshold?  
At 100 % completeness, how many proteins do you get? Why is this rarely used?

### Exercise B — Outlier threshold
Change `N_METHODS_THRESHOLD` to 1 and 5.  
How many samples are removed at each setting?  
What is the risk of being too conservative (threshold=1) vs. too strict (threshold=5)?

### Exercise C — Batch correction comparison
Run the PC × factor associations analysis on the `quant_combat` matrix.  
Does the plate association drop below significance (−log10(p) < 1.3)?  
Does the group association increase?

### Exercise D — Your own data
Replace `demo_parquet` with a path to your own Spectronaut export.  
You will need to ensure:
1. The parquet has `PG_ProteinAccessions` as the first column
2. Sample columns contain log2 intensities (Spectronaut default)
3. A metadata CSV with at least a `plate` column exists

---
*Pipeline source code: github.com/nigelkurgan/ddea-proteomics-course*